# 📊 Automated MLOps Report
This notebook is automatically executed by `dvc repro`.
It generates visualizations for the current state of the data AND the Best Model.

In [ ]:
import pandas as pd
import yaml
import seaborn as sns
import matplotlib.pyplot as plt
import os
import joblib
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

# Establish directories
os.makedirs("reports/figures", exist_ok=True)

## 1. Load Data & Rules

In [ ]:
# Load Parameters
with open("params.yaml") as f:
    params = yaml.safe_load(f)

print(f"🔹 Loaded parameters from params.yaml: {params}")

# Load Processed Data (PCA)
X_train = pd.read_csv("data/processed/pcos_train.csv")
y_train = pd.read_csv("data/processed/pcos_y_train.csv")

# Combine for plotting stats
df = X_train.copy()
df['target'] = y_train.values
print(f"🔹 Data Loaded. Shape: {df.shape}")

## 2. SMOTE Results (Class Distribution)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette='viridis')
plt.title("Class Distribution after SMOTE (Training Set)")
plt.savefig("reports/figures/class_distribution.png")
plt.show()
print("✅ Saved reports/figures/class_distribution.png")

## 3. PCA Visualization (Separability)

In [ ]:
# Assuming first two columns are PC1 and PC2 (since it is PCA output)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=df.iloc[:, 0], y=df.iloc[:, 1], hue=df['target'], alpha=0.7, palette='coolwarm')
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"PCA Visualization (Top 2 Components)")
plt.savefig("reports/figures/pca_scatter.png")
plt.show()
print("✅ Saved reports/figures/pca_scatter.png")

## 4. Feature Correlation

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.drop('target', axis=1).corr(), cmap='coolwarm', annot=False)
plt.title("Feature Correlation Matrix (PCA Components)")
plt.savefig("reports/figures/correlation_matrix.png")
plt.show()
print("✅ Saved reports/figures/correlation_matrix.png")

## 5. Best Model Performance (Confusion Matrix & ROC)

In [ ]:
# Load Test Data and Best Model
try:
    model = joblib.load("app/models/best_model.pkl")
    X_test = pd.read_csv("data/processed/pcos_test.csv")
    y_test = pd.read_csv("data/processed/pcos_y_test.csv").values.ravel()

    # Predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')
    plt.title("Confusion Matrix (Best Model)")
    plt.savefig("reports/figures/confusion_matrix.png")
    plt.show()
    print(" Saved reports/figures/confusion_matrix.png")

    # ROC Curve (if supported)
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        
        plt.figure()
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic (ROC)')
        plt.legend(loc="lower right")
        plt.savefig("reports/figures/roc_curve.png")
        plt.show()
        print(" Saved reports/figures/roc_curve.png")
except FileNotFoundError:
    print(" Model or Test data not found yet. Run 'dvc repro' first.")